💡 **Environment:** `clamp-analyses`

# Supplementary Figure 5: CLAMPbase trait recovery at K = 1,728


In [ ]:
suppressPackageStartupMessages({
  library(here)
})
source(here("scripts/panels/style.R"))

In [ ]:
set.seed(123)
base <- here("output/03_model_biology/02_archs4")
paths <- c(pathway_coverage=file.path(base,"00_coverage/coverage_long.csv"),
           trait_coverage=file.path(base,"00_coverage/coverage_trait_recovery.csv"),
           trait_full_compendia=file.path(base,"00_coverage/final_trait_recovery.csv"),
           pathway_saturation=file.path(base,"01_saturation/saturation_long.csv"),
           trait_saturation=file.path(base,"01_saturation/saturation_trait_recovery_k1728.csv"))
out <- nm_sources(5,paths)
cov <- fread(paths[1])[database!="reactome"]
traits <- fread(paths[2]); finals <- fread(paths[3]); sat <- fread(paths[4])[database!="reactome"]; tsat <- fread(paths[5])
cols <- unlist(yaml::read_yaml(here("config.yaml"))$MODEL_COLORS)[c("CLAMPbase")]
compendium_cols <- unlist(yaml::read_yaml(here("config.yaml"))$COMPENDIUM_COLORS)
fraction_cols <- unlist(yaml::read_yaml(here("config.yaml"))$ARCHS4_FRACTION_COLORS)
models <- c("CLAMPbase")
# 100% is kept only where its replicates are distinct models (see Figure 3 notebook).
keep_distinct_100 <- function(d) d[fraction<100 | model %in% d[fraction==100,.(n=uniqueN(recovered)),by=model][n>1,model]]
paths_cov <- cov[dataset=="archs4",.(recovered=sum(recovered_pathways),eligible=sum(eligible_pathways)),by=.(model,fraction,seed)]
trait_cov <- traits[,.(model,fraction,seed,recovered=recovered_traits,eligible=eligible_traits)]
paths_cov <- keep_distinct_100(paths_cov); trait_cov <- keep_distinct_100(trait_cov)
psat <- sat[dataset=="archs4",.(recovered=sum(recovered_pathways),eligible=sum(eligible_pathways)),by=.(model,fraction,k,seed)]
tsat <- tsat[,.(model,fraction,k,seed,recovered=recovered_traits,eligible=eligible_traits)]
recovery_plot <- function(d,model_name,metric,saturation=FALSE,paired=FALSE,all_tests=FALSE,ref_fracs=c(50)) {
  d <- copy(d[model==model_name]);d[,fraction_label:=factor(paste0(fraction,"%"),levels=paste0(sort(unique(fraction)),"%"))]
  totals <- d[,.(recovered=median(recovered),eligible=median(eligible),minimum=min(recovered)),by=fraction_label]
  span <- max(1,diff(range(d$recovered)))
  totals[,label:=sprintf("%s\n(%.0f%%)",format(round(recovered),big.mark=",",trim=TRUE),100*recovered/eligible)]
  ref_frac <- max(d$fraction)
  refs <- if(all_tests) setdiff(unique(d$fraction),ref_frac) else union(ref_fracs, if(ref_frac==100) 75)
  # Widest span on top, so a long bracket never crosses the boxes it skips.
  refs <- sort(intersect(refs,unique(d$fraction)),decreasing=TRUE)
  tests <- rbindlist(lapply(refs,function(f) {
    ref <- d[fraction==ref_frac][order(seed)]; cand <- d[fraction==f][order(seed)]
    if(paired) {common<-intersect(ref$seed,cand$seed);ref<-ref[seed%in%common];cand<-cand[seed%in%common]}
    if(nrow(ref)<2 || nrow(cand)<2)return(NULL)
    data.table(fraction=f,p=t.test(ref$recovered,cand$recovered,paired=paired,var.equal=!paired,alternative="greater")$p.value,
               delta=mean(ref$recovered)-mean(cand$recovered))
  }))
  tests[,`:=`(x=match(paste0(fraction,"%"),levels(d$fraction_label)),xend=match(paste0(ref_frac,"%"),levels(d$fraction_label)),
               y=max(d$recovered)+span*(.16+.20*seq_len(.N)),label=sprintf("P = %s; %+.0f",nm_p(p),delta))]
  fwrite(tests,file.path(out,paste0(model_name,"_",metric,if(saturation)"_saturation" else "_coverage","_tests.csv")))
  p <- ggplot(d,aes(fraction_label,recovered,fill=fraction_label,colour=fraction_label))+
    geom_boxplot(alpha=.35,width=.6,outlier.shape=NA,linewidth=.25)+
    geom_point(position=position_jitter(width=.08,height=0),size=.7,alpha=.8)+
    scale_fill_manual(values=fraction_cols,guide="none")+scale_colour_manual(values=fraction_cols,guide="none")+
    geom_text(data=totals,aes(x=fraction_label,y=minimum-span*.10,label=label),inherit.aes=FALSE,vjust=1,size=5.5/.pt,lineheight=.95)+
    geom_segment(data=tests,aes(x=x,xend=xend,y=y,yend=y),inherit.aes=FALSE,linewidth=.25,colour="black")+
    geom_text(data=tests,aes(x=(x+xend)/2,y=y+span*.025,label=label),inherit.aes=FALSE,vjust=0,size=5.5/.pt)+
    scale_y_continuous(expand=expansion(mult=c(.20,.12)))+
    labs(x="Studies used",y=paste("Recovered",metric),title=model_name)+nm_theme()
  if(!saturation)p<-p+stat_summary(aes(group=1),fun=median,geom="line",linetype="dashed",linewidth=.3,colour="#222222")
  p
}
cross <- cov[fraction==100,.(recovered=sum(recovered_pathways)),by=.(dataset,model,seed)]
cross_plot <- function(d,m,metric) {
  d<-copy(d[model==m & dataset!="recount2"]);d[,dataset:=factor(dataset,levels=c("archs4","gtex"),labels=c("ARCHS4","GTEx"))]
  lab<-d[,.(recovered=round(median(recovered))),by=dataset]
  boxed<-d[dataset %in% d[,.(n=uniqueN(recovered)),by=dataset][n>1,dataset]]
  p<-ggplot(d,aes(dataset,recovered,colour=dataset))
  if(nrow(boxed)) p<-p+geom_boxplot(data=boxed,aes(fill=dataset),alpha=.35,width=.5,outlier.shape=NA,linewidth=.25)
  p+
    geom_point(size=1.2,position=position_jitter(width=.06,height=0))+
    geom_text(data=lab,aes(label=format(recovered,big.mark=",",trim=TRUE)),vjust=-1.4,size=5.5/.pt,colour="black")+
    scale_fill_manual(values=compendium_cols,guide="none")+
    scale_colour_manual(values=compendium_cols,guide="none")+
    scale_y_continuous(expand=expansion(mult=c(.1,.25)))+labs(x=NULL,y=paste("Recovered",metric),title=m)+nm_theme()
}
saturation_plot <- function(m) {
 d<-copy(psat[model==m & fraction<100]);d[,`:=`(K=factor(k),fraction_label=factor(paste0(fraction,"%"),levels=paste0(sort(unique(fraction)),"%")))]
 ggplot(d,aes(K,recovered,fill=fraction_label,colour=fraction_label))+
   geom_boxplot(position=position_dodge(.85),alpha=.35,outlier.shape=NA,linewidth=.25)+
   geom_point(position=position_jitterdodge(jitter.width=.1,dodge.width=.85),size=.5,show.legend=FALSE)+
   stat_summary(aes(group=fraction_label),fun=mean,geom="line",position=position_dodge(.85),linetype="dashed",linewidth=.3)+
   scale_fill_viridis_d(end=.9,direction=-1)+scale_colour_viridis_d(end=.9,direction=-1)+
   labs(x="Latent variables (K)",y="Recovered pathways",title=m,fill="Studies used",colour="Studies used")+nm_theme()+
   theme(legend.position="bottom")+guides(fill=guide_legend(nrow=1),colour=guide_legend(nrow=1))
}
# CLAMPbase trait recovery at K = 1,728 across ARCHS4 training fractions (not shown in Figure 3).
fig <- recovery_plot(tsat,"CLAMPbase","traits",TRUE,FALSE,FALSE,c(25,50))+labs(title=NULL,x="Training studies used (K = 1,728)")
nm_export(list(fig),5,"Trait recovery at fixed rank",height=75)
